# Dark Count Rate Analysis

In [2]:
%load_ext autoreload
%autoreload 2

## Import packages

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('../')
import pylars

In [4]:
import pylars.utils.input
import pylars.utils.output
from pylars.analysis.darkcount import *
from pylars.plotting.plotprocessed import *
from pylars.plotting.plotwaveforms import *

## Setup the run

In [5]:
run_number = 2
main_data_path = '../kadev/data/'
F_amp = 1
ADC_model = 'v1724'

In [6]:
base_run = pylars.utils.input.run(
    run_number,
    main_data_path,
    F_amp,
    ADC_model,
)

### initiate processor

In [7]:
sigma_level = 2
baseline_samples = 14_000

In [8]:
process = pylars.processing.rawprocessor.run_processor(
    base_run,
    'simple',
    sigma_level,
    baseline_samples,
)

### initiate dataset

in contrast to the Breakdown Voltage computation with only a dataset class and seperate analysis functions the Dark Count Rate analysis is structured in multiple different classes.
- DCR Run class to handle multiple datasets
- DCR Dataset (object-) class to manage a dataset, inherits DCR Analysis
- DCR Analysis class containing "class methods" for analysis (DCR, CTP, gain computations and so on)

The DCR run class still requires a base run from "pylars.utils.input.run()", it doesn't replace it!

In [9]:
temperature = 170
module = 0
channel = 'wf0'
processor = process

In [10]:
dcr_run = DCR_run(
    base_run,
    processor,
)

#### DCR Run class

contains
- get_run_temperatures(): list all temperatures in the available data folder

In [11]:
dcr_run.get_run_temperatures()

array([170.])

- initialize_results_df(): creates an empty dataframe structure expected for this DCR analysis

In [12]:
dcr_run.initialize_results_df()
dcr_run.results_df

,T,V,SPE_area,SPE_area_error,Gain,Gain_error,SPE_res,SPE_res_error,DCR,DCR_error,CTP,CTP_error


- define_run_SiPM_config(sensor_area: float = 12 * 12): set SiPM config, area in mm^2 (livetime commented out in source code)

In [13]:
dcr_run.define_run_SiPM_config(sensor_area = 6 * 6)
dcr_run.SiPM_config

{'sensor_area': 36}

- set_standard_cuts_run(parameters): convenient way to standardise cuts throughout the DCR analysis for the specific run

##### Parameters are:
  
cut_area_min: float = 5,

cut_area_max: float = 1e6,

cut_length_min: int = 4,

cut_length_max: int = 70,

cut_n_pulses_min: float = 0,

cut_n_pulses_max: float = 2

In [15]:
dcr_run.set_standard_cuts_run()
area_min = dcr_run.cut_area_min
print(f'The applied cut regarding minimum area is : {area_min}.')


The applied cut regarding minimum area is : 5.


- load_dataset(): **requires pre-processed data** for all raw files of the DCR run, creates a DCR Dataset object and loads the pre-processed data

It uses "DCR_dataset()" to create the dataset object and the "load_processed_data()" function of the DCR dataset class, which is elaborated further below.

In [ ]:
dcr_run.load_dataset(
    temp,
    module,
    channel,
)

- compute_properties_of_ds(): **requires pre-processed data**

uses "load_dataset()" to create DCR Dataset object, then applies the standard cuts defined for the run to the object.

Additionally uses "compute_properties_of_dataset_amplitude_based()" or "compute_properties_of_dataset()" from the DCR Dataset class and returns a pd.DataFrame with additional columns for module and channel.

In [ ]:
dcr_run.compute_properties_of_ds(
    temp,
    module,
    channel,
    amplitude_based, # bool = False
)

- compute_properties_of_run(): **requires pre-processed data**

uses "pylars.utils.common.get_channel_list(self.process)" to pull a list of the available channels with corresponding module numbers and runs "compute_properties_of_ds()" for each combination.

Saves the result dataframes as a concatonated dataframe in self.results_df

In [24]:
print('found (module, channel) tuples: ', pylars.utils.common.get_channel_list(dcr_run.process))
print('found temperatures: ', dcr_run.temperatures)

found (module, channel) tuples:  [(np.int64(0), 'wf0')]
found temperatures:  [170.]


In [25]:
dcr_run.results_df

,T,V,SPE_area,SPE_area_error,Gain,Gain_error,SPE_res,SPE_res_error,DCR,DCR_error,CTP,CTP_error


- save_results(): **requires an existing dataframe with results**

saves the result dataframe to a hdf5 .h5 file.

In [30]:
custom_name = "test"
name = "test"

In [ ]:
dcr_run.save_results(
    custom_name,
)

- load_results(): **requires an existing .h5 file**

loads the result from a saved hdf5 .h5 file and returns a dataframe

In [ ]:
dcr_run.load_results(
    name,
)

#### DCR Analysis class

is inherited by DCR Dataset class, which is why it's introduced first.

Contains "@classmethod"s
- get_1pe_value_fit()
- get_1pe_rough() and
- get_DCR_above_threshold_values()

As well as the "@staticmethod"
- get_DCR()

@classmethod: when called the objects class is passed as self instead of the object
@staticmethod: passes no self at all, only the parameters explicitly passed

In [31]:
dcr_analysis = DCR_analysis()

- get_DCR_above_threshold_values(): computes DCR over pulse-area threshold

    1. applies length cuts do the dataframe
    2. creates a histogram for the remaining pulses
    3. maps centers of each bin
    4. computes DCR above each threshold by summing ob the counts of the bins above the threshold
    5. returns either
       - 'values': (histogram area, DCR values)
       - 'spline': UnivariateSpline object - a smooth interpolation + derivative
       - 'interp1d': interp1d object - a linear interpolation

In [ ]:
dcr_analysis.get_DCR_above_threshold_values(
    df,
    length_cut_min, # int = 5
    length_cut_max, # int = 80
    bins,           # Union[int,list] = 200
    output,         # str = 'values'
)

- get_1pe_value_fit(): provides a rough estimate of the 1PE area without fitting the peak

    - at low pulse areas the DCR is dominated by baseline noise
    - if the threshold approaches the 1PE area the DCR should drop rapidly
      $\rightarrow$ the steepest negative slope of the DCR over threshold curve marks the transition from noise- to signal-dominated events

      $\rightarrow$ uses this point as the rough 1PE area

      1. uses "get_DCR_above_threshold_values()" to calculate the DCR over threshold
      2. computes and fits its derivative
      3. identifies the most negative value of the smoothed derivative (with or without scipys find_peaks)
      4. returns (fitted areas, corresponding DCRs, DCR derivative x, DCR derivative y, determined 1PE area)


Note: the function avoids unusually large first peaks via 'peaks_area_values[0] > 2000 * 200' in order to prevent selecting a noise-related feature instead of the first 1PE peak.

In [ ]:
length_cut_min = 5
length_cut_max = 80
bins = 200

In [ ]:
dcr_analysis.get_1pe_rough(
    df,
    length_cut_min,       # int
    length_cut_max,       # int
    bins,                 # Union[int, list] = 200
    use_scipy_find_peaks, # bool = False
)

- get_1pe_value_fit(): locates and fits the 1PE peak to extract its area

    1. uses "get_1pe_rough()" to estimate the approximate area where the first photoelectron peak begins

    2. applies length cuts to filter out electronic noise etc.

    3. creates a histogram of the pulse areas around the "get_1pe_rough()" estimation with 0.5 $\mu_{rough}\leq areas \leq 1.5\mu_{rough}$, this way the fit is not considering the pedestal or multi PE peaks at higher areas

    4. fit the histogram with $f(x) = \frac{A}{\sqrt{2\pi\sigma^2}}\cdot exp[-\frac{(x-\mu)^2}{2\sigma^2}]$, with

        - $A$: number of events in the 1PE peak
        - $\mu$: fitted 1PE area
        - $\sigma$: standard deviation $\rightarrow$ resolution of the 1PE peak

        initial guesses are fixed to $p0=(2000, \mu_{rough}, 0.05 \cdot \mu_{rough})$
 
    5. returns $A, \mu, \sigma \; \& \; cov$ such that one can approximate PE counts via $N_{PE} \approx \frac{pulse\;area}{\mu_{1PE}}$ or the 1PE resolution $\frac{\sigma}{\mu}$

In [ ]:
dcr_analysis.get_1pe_value_fit(
    df,
    length_cut_min,       # int = 5
    length_cut_max,       # int = 80
    plot,                 # Union[bool, str] = False
    use_scipy_find_peaks, # bool = False
)

- get_DCR(): computes the DCR and CTP, **requires previously computed 1PE area**

  1. sets thresholds
     - $area \; > 0.5\;PE$: contain at least one PE
     - $area \; > 1.5\;PE$: contain more than one PE (in a DCR run this is probabl 1PE + CT)
  2. applies pulse length cuts
  3. selects the events falling into the threshold categories and computes DCR as
     $$DCR = \frac{N}{A_{SiPM}t},$$

     where $N$ is either $N_{>0.5PE}$ or $N_{>1.5PE}$, $A_{SiPM}$ is the effective sensor area and $t$ the aquisition time.
  4. computes statistical uncertainty assuming Poisson statistics ($\sigma_N = \sqrt{N}$) such that
     $$\sigma_{DCR} = \frac{\sqrt{N}}{A_{SiPM}t}.$$

  5. estimates CTP as the ratio
     $$CTP = \frac{N_{>1.5PE}}{N_{>0.5PE}}$$
  6. and its uncertainty by error propagation, considering that $N_{>0.5PE}$ includes $N_{>1.5PE}$
  7. returns DCR, DCR error, CTP and CTP error

Note: the DCR here is normalised according to the effective sensor area!

In [ ]:
dcr_analysis.get_DCR(
    length_cut_min, # int
    length_cut_max, # int
    spe_area,       # float
    sensor_area,    # float
    t,              # float
)

#### DCR Dataset Class

inherits the DCR Analysis Class as it is initialised as "DCR_dataset(DCR_analysis)", thus classmethods will return the underlying "DCR_analysis" object as self when used on a DCR_dataset.

